# MLP (TensorFlow)

Thesis experiment: feed-forward network, rolling-origin evaluation, three feature sets. Needs `tensorflow`.

In [ ]:
import pandas as pd
import numpy as np
from math import sqrt
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from keras.layers import Dense
from keras.models import Sequential

In [ ]:
from energyforecast.data import load_dataset

data = load_dataset("../../data")

In [ ]:
def plot_model_rmse_and_loss(history):

    #evaluating train and validation accuracies and losses

    train_rmse = history.history['root_mean_squared_error']
    #val_rmse = history.history['val_root_mean_squared_error']

    train_loss = history.history['loss']
    #val_loss = history.history['val_loss']

    #visualizing epochs vs. train and validation accuracies and losses

    plt.figure(figsize=(20, 10))
    plt.subplot(1, 2, 1)
    plt.plot(train_rmse, label='Training RMSE')
    #plt.plot(val_rmse, label='Validation RMSE')
    plt.legend()
    plt.title('Epochs / Training RMSE')

    plt.subplot(1, 2, 2)
    plt.plot(train_loss, label='Training Loss')
    #plt.plot(val_loss, label='Validation Loss')
    plt.legend()
    plt.title('Epochs / Training Loss')

    plt.show()

def plot_preds_vs_actual(true, preds):
    plt.figure(figsize=(12,6))
    plt.plot(true, label='Real')
    plt.plot(preds, label='Predicted', color='red')
    plt.title('Predicted vs Real Values')
    plt.title('Actual vs Predicted Values')
    plt.xlabel('Time')
    plt.ylabel('Total Aggregated')
    plt.legend()
    plt.show()

In [ ]:
n_train = 35064

features = ['total_aggregated']
feature_array = data[features].values

# Initialize the scaler
scaler = StandardScaler()

scaler.fit(feature_array[:n_train].reshape(-1,1))
scaled_array = scaler.transform(feature_array)

# Create a DataFrame from the observations
df = pd.DataFrame(scaled_array, columns=['total_aggregated'])

# Create 24 lags and store them as new columns in the DataFrame
for i in range(1, 25):
    df[f'lag_{i}'] = df['total_aggregated'].shift(i)

# Interpolate missing values
df = df.bfill()
# Define the number of train and test observations

# Separate the lags (features) and the observation (target)
#X = df.drop('total_aggregated', axis=1)
X = df['lag_24']
y = df['total_aggregated']

# Separate the features into training and test sets
X_train = X[:n_train]
X_test = X[n_train:]

y_train, y_test = y[:n_train], y[n_train:]

In [ ]:
truth = feature_array[-len(y_test):]

In [ ]:
loss = tf.keras.losses.MeanSquaredError()
metric = [tf.keras.metrics.RootMeanSquaredError()]
lr_schedule = tf.keras.callbacks.LearningRateScheduler(
              lambda epoch: 1e-4 * 10**(epoch / 10))
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss',patience=3)
optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=0.003)

In [ ]:
# Define the model
mlp = tf.keras.models.Sequential([
    Dense(10, activation='relu', input_shape=(1,)),

    Dense(10, activation='relu'),

    Dense(1, activation='linear')
])

mlp.summary()

mlp.compile(optimizer=optimizer, loss=loss, metrics = metric)

In [ ]:
history_X = X_train.copy()
history_y = y_train.copy()

In [ ]:
window = 24*365*3  # training window length
n_forecast_steps = 24  # forecast horizon

In [ ]:
history_X = X_train.copy()  # history starts as the training window
history_y = y_train.copy()

predictions = []  # forecasts
errors = []  # RMSE per period
days = 1  # period counter
for i in range(0, len(X_test), n_forecast_steps):
    t_end = i + n_forecast_steps  # forecast the block [i, i + n_forecast_steps)

    if t_end > len(X_test):
        t_end = len(X_test)
    X_test_block = X_test[i:t_end]
    y_test_block = y_test[i:t_end]
    truth_block = truth[i:t_end]

    print('training and predicting for period', days)
    mlp.fit(history_X, history_y, epochs=100, verbose=0, batch_size=168, callbacks=[early_stopping])  # fit on the current history, up to 100 epochs
    # early stopping ends the fit when the loss
    # stops decreasing for `patience` epochs

    yhat = mlp.predict(X_test_block)  # predict and map back to MW

    yhat_rescaled = scaler.inverse_transform(yhat)

    rmse = sqrt(np.mean((truth_block - yhat_rescaled)**2))  # RMSE of this block
    errors.append(rmse)
    predictions.extend(yhat_rescaled)

    print('RMSE:', rmse)

    days = days + 1

    # Update the history with the latest "known" data
    history_X = pd.concat([history_X[n_forecast_steps:], X_test_block])  # slide the history forward by one block:
    history_y = pd.concat([history_y[n_forecast_steps:], y_test_block])  # drop the oldest hours, append the observed ones

In [ ]:
#y_pred_inv = scaler.inverse_transform(predictions)
#
## Plot the actual and predicted values
plot_preds_vs_actual(truth, predictions)
#rmse = sqrt(np.mean((truth - predictions)**2))

In [ ]:
res = pd.DataFrame()
res['y_true'] = pd.Series(truth.flatten())
res['y_pred'] = predictions

rmse = sqrt(np.mean((res.y_true - res.y_pred)**2))
print('RMSE:', rmse)

## Weekend dummies

In [ ]:
features = ['total_aggregated', 'saturday', 'sunday']
feature_array = data[features].values
#window_size = 24

# Fit Scaler only on Training target values
target_scaler = StandardScaler()
target_scaler.fit(feature_array[:n_train, 0].reshape(-1,1))

# Transform both Training and Test data
scaled_feature = target_scaler.transform(feature_array[:, 0].reshape(-1, 1))

scaled_array = np.copy(feature_array)
scaled_array[:, 0] = scaled_feature.flatten()

# Create a DataFrame from the observations
df = pd.DataFrame(scaled_array, columns=['total_aggregated', 'sat', 'sun'])

# Create 24 lags and store them as new columns in the DataFrame
for i in range(1, 25):
    df[f'lag_{i}'] = df['total_aggregated'].shift(i)

# Interpolate missing values
df = df.bfill()
# Define the number of train and test observations

# Separate the lags (features) and the observation (target)
X = df[['lag_1', 'sat', 'sun']]
y = df['total_aggregated']

# Separate the features into training and test sets
X_train = X[:n_train]
X_test = X[n_train:]

y_train, y_test = y[:n_train], y[n_train:]

In [ ]:
# Define the model
mlp2 = tf.keras.models.Sequential([
    Dense(10, activation='relu', input_shape=(3,)),
    Dense(10, activation='relu'),
    #Dense(1, activation='relu'),
    Dense(1, activation='linear')
])

mlp2.summary()
mlp2.compile(optimizer=optimizer, loss=loss, metrics = metric)

In [ ]:
history_X = X_train.copy()  # history starts as the training window
history_y = y_train.copy()

predictions2 = []  # forecasts
errors2 = []  # RMSE per period
days = 1  # period counter
for i in range(0, len(X_test), n_forecast_steps):
    t_end = i + n_forecast_steps  # forecast the block [i, i + n_forecast_steps)

    if t_end > len(X_test):
        t_end = len(X_test)
    X_test_block = X_test[i:t_end]
    y_test_block = y_test[i:t_end]
    truth_block = truth[i:t_end]

    print('training and predicting for period', days)
    mlp2.fit(history_X, history_y, epochs=100, verbose=0, batch_size=168, callbacks=[early_stopping])  # fit on the current history, up to 100 epochs
    # early stopping ends the fit when the loss
    # stops decreasing for `patience` epochs

    yhat = mlp2.predict(X_test_block)  # predict and map back to MW

    yhat_rescaled = scaler.inverse_transform(yhat)

    rmse = sqrt(np.mean((truth_block - yhat_rescaled)**2))  # RMSE of this block
    errors2.append(rmse)
    predictions2.extend(yhat_rescaled)

    print('RMSE:', rmse)

    days = days + 1

    # Update the history with the latest "known" data
    history_X = pd.concat([history_X[n_forecast_steps:], X_test_block])  # slide the history forward by one block:
    history_y = pd.concat([history_y[n_forecast_steps:], y_test_block])  # drop the oldest hours, append the observed ones

In [ ]:
## Plot the actual and predicted values
plot_preds_vs_actual(truth, predictions2)

In [ ]:
res['y_pred2'] = predictions2

rmse2 = sqrt(np.mean((res.y_true - res.y_pred2)**2))
print('RMSE:', rmse2)

## Business-hour dummy

In [ ]:
features = ['total_aggregated', 'business_hour']
feature_array = data[features].values

# Fit Scaler only on Training target values
target_scaler = StandardScaler()
target_scaler.fit(feature_array[:n_train, 0].reshape(-1,1))

# Transform both Training and Test data
scaled_feature = target_scaler.transform(feature_array[:, 0].reshape(-1, 1))

scaled_array = np.copy(feature_array)
scaled_array[:, 0] = scaled_feature.flatten()

# Create a DataFrame from the observations
df = pd.DataFrame(scaled_array, columns=['total_aggregated', 'business_hour'])

# Create 24 lags and store them as new columns in the DataFrame
for i in range(1, 25):
    df[f'lag_{i}'] = df['total_aggregated'].shift(i)

# Interpolate missing values
df = df.bfill()
# Define the number of train and test observations

# Separate the lags (features) and the observation (target)
#X = df.drop('total_aggregated', axis=1)
X = df[['lag_1', 'business_hour']]
y = df['total_aggregated']

# Separate the features into training and test sets
X_train = X[:n_train]
X_test = X[n_train:]

y_train, y_test = y[:n_train], y[n_train:]

In [ ]:
# Define the model
mlp3 = tf.keras.models.Sequential([
    Dense(10, activation='relu', input_shape=(2,)),
    Dense(10, activation='relu'),
    Dense(1, activation='linear')
])

mlp3.summary()
mlp3.compile(optimizer=optimizer, loss=loss, metrics = metric)

In [ ]:
window = 24*365*3  # training window length
n_forecast_steps = 24*7  # forecast horizon

history_X = X_train.copy()  # history starts as the training window
history_y = y_train.copy()

predictions3 = []  # forecasts
errors3 = []  # RMSE per period
days = 1  # period counter
for i in range(0, len(X_test), n_forecast_steps):
    t_end = i + n_forecast_steps  # forecast the block [i, i + n_forecast_steps)

    if t_end > len(X_test):
        t_end = len(X_test)
    X_test_block = X_test[i:t_end]
    y_test_block = y_test[i:t_end]
    truth_block = truth[i:t_end]

    print('training and predicting for period', days)
    mlp3.fit(history_X, history_y, epochs=100, batch_size=168, verbose=0, callbacks=[early_stopping])  # fit on the current history, up to 100 epochs
    # early stopping ends the fit when the loss
    # stops decreasing for `patience` epochs

    yhat = mlp3.predict(X_test_block)  # predict and map back to MW

    yhat_rescaled = scaler.inverse_transform(yhat)

    rmse = sqrt(np.mean((truth_block - yhat_rescaled)**2))  # RMSE of this block
    errors3.append(rmse)
    predictions3.extend(yhat_rescaled)

    print('RMSE:', rmse)

    days = days + 1

    # Update the history with the latest "known" data
    history_X = pd.concat([history_X[n_forecast_steps:], X_test_block])  # slide the history forward by one block:
    history_y = pd.concat([history_y[n_forecast_steps:], y_test_block])  # drop the oldest hours, append the observed ones

In [ ]:
## Plot the actual and predicted values
plot_preds_vs_actual(truth, predictions3)

In [ ]:
res['y_pred3'] = predictions3
rmse3 = sqrt(np.mean((truth - predictions3)**2))
print('RMSE:', rmse3)

In [ ]:
rmse1 = sqrt(np.mean((res.y_true - res.y_pred)**2))
print('Root Mean Squared Error just TS:', rmse1)
rmse2 = sqrt(np.mean((res.y_true - res.y_pred2)**2))
print('Root Mean Squared Error WE dummies:', rmse2)
rmse3 = sqrt(np.mean((res.y_true - res.y_pred3)**2))
print('Root Mean Squared Error BH dummy:', rmse3)